In [ ]:
from neo4j import GraphDatabase
import pandas as pd

URI = "neo4j://127.0.0.1:7687"
AUTH_USER = "neo4j"
AUTH_PASSWORD = "master2025"
DATABASE = "llmakg"

driver = GraphDatabase.driver(URI, auth=(AUTH_USER, AUTH_PASSWORD))

In [ ]:
BRIDGE_QUERY = """
MATCH (c1:Chunk)-[:MENTIONS]->(a:__Entity__),
      (c1)-[:MENTIONS]->(x:__Entity__),
      (c2:Chunk)-[:MENTIONS]->(x),
      (c2)-[:MENTIONS]->(b:__Entity__)
WHERE c1 <> c2
  AND a <> b AND a <> x AND b <> x
  AND c1.file_name <> c2.file_name
  AND NOT EXISTS {
    MATCH (c:Chunk)-[:MENTIONS]->(a),
          (c)-[:MENTIONS]->(b)
  }
RETURN
  a.id  AS A,
  x.id  AS bridge,
  b.id  AS B,
  c1.chunk_id AS chunk1,
  c1.file_name AS file1,
  c2.chunk_id AS chunk2,
  c2.file_name AS file2
LIMIT 100
"""
with driver.session(database=DATABASE) as session:
    rows = session.run(BRIDGE_QUERY)
    bridge_df = pd.DataFrame([r.data() for r in rows])

bridge_df.head()
